# PatchCore DINOv2 ViT-B/14 (`x224`)

This notebook is the canonical evaluation workflow for PatchCore built on a frozen DINOv2 ViT-B/14 backbone.

DINOv2 uses the same ViT-B architecture as the supervised baseline but is pretrained with DINO self-distillation on 142M curated images. Its patch tokens retain strong local spatial structure throughout all layers, unlike supervised ViT which collapses to class-discriminative representations at deeper blocks. This makes it a natural backbone for patch-level anomaly detection.

**No training is performed.** The backbone is fully frozen. The only compute is one forward pass over the training set to build the memory bank.

**Control flags (set in the Configuration cell):**
- `FORCE_REBUILD_SCORES = False` - loads saved scores from `scores.npz`. Set `True` to rebuild the memory bank and re-score.
- `FORCE_RERUN_UMAP = False` - displays the saved UMAP figure. Set `True` to reproject.

Artifacts written by this notebook:
- [`experiments/anomaly_detection/patchcore/dinov2_vit_b14/x224/main/artifacts/checkpoints`](experiments/anomaly_detection/patchcore/dinov2_vit_b14/x224/main/artifacts/checkpoints)
- [`experiments/anomaly_detection/patchcore/dinov2_vit_b14/x224/main/artifacts/plots`](experiments/anomaly_detection/patchcore/dinov2_vit_b14/x224/main/artifacts/plots)
- [`experiments/anomaly_detection/patchcore/dinov2_vit_b14/x224/main/artifacts/results`](experiments/anomaly_detection/patchcore/dinov2_vit_b14/x224/main/artifacts/results)

## Comparison baseline

| Backbone | Patch size | Tokens/image | Pretraining | F1 | AUROC | AUPRC |
|---|---|---|---|---|---|---|
| ViT-B/16 augreg (frozen) | 16 | 196 | Supervised ImageNet-21k | 0.595 | 0.956 | 0.671 |
| ViT-B/14 DINOv2 (frozen) | 14 | 256 | DINO self-distillation, 142M images | TBD | TBD | TBD |

## Setup

This cell installs or checks optional notebook dependencies before the rest of the workflow runs.

In [ ]:
import importlib.util
import subprocess
import sys
for pkg in ['timm', 'tqdm']:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('timm + tqdm ready')

## Imports

In [ ]:
import os, gc, json, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, average_precision_score, precision_recall_curve, f1_score, precision_score, recall_score
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_CUDA = DEVICE.type == 'cuda'
if USE_CUDA:
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('high')
print('Device:', DEVICE)
if USE_CUDA:
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}  VRAM: {p.total_memory / 1000000000.0:.1f} GB')
from IPython.display import display, Image as IPImage

## Configuration

This cell resolves the repo root, defines experiment settings, and points all outputs to the local artifact folders.

**DINOv2 note:** patch size 14 gives 256 tokens per image at 224x224 (vs 196 for ViT-B/16). All other PatchCore settings are held identical to the frozen ViT-B/16 baseline for a controlled comparison.

In [ ]:
try:
    from pathlib import Path
    cwd = Path.cwd().resolve()
    PROJECT_ROOT = None
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'src' / 'wafer_defect').exists() and (candidate / 'configs').exists():
            PROJECT_ROOT = candidate
            break
    if PROJECT_ROOT is None:
        raise RuntimeError('Could not locate repo root containing src/wafer_defect and configs/')
    FORCE_REBUILD_SCORES = False
    FORCE_RERUN_UMAP = False
    DATA_PATH = str(PROJECT_ROOT / 'data' / 'raw' / 'LSWMD.pkl')
    IMAGE_SIZE = 224
    TRAIN_NORMAL_N = 40000
    TUNE_NORMAL_N = 5000
    TEST_NORMAL_N = 5000
    TEST_DEFECT_N = 250
    BACKBONE_NAME = 'vit_base_patch14_dinov2.lvd142m'
    VIT_FEATURE_BLOCK = 9
    PATCH_EMBED_DIM = 128
    MEMORY_BANK_MAX = 600000
    SCORE_CHUNK = 512
    PATCHCORE_NN_K = 3
    TOPK_PATCH_RATIO = 0.1
    BATCH_SIZE = 128
    NUM_WORKERS = 0
    THRESHOLD_QUANTILE = 0.95
    ARTIFACT_DIR = str(PROJECT_ROOT / 'experiments/anomaly_detection/patchcore/dinov2_vit_b14/x224/main/artifacts')
    CHECKPOINTS_DIR = os.path.join(ARTIFACT_DIR, 'checkpoints')
    PLOTS_DIR = os.path.join(ARTIFACT_DIR, 'plots')
    RESULTS_DIR = os.path.join(ARTIFACT_DIR, 'results')
    MODEL_EXPORT_PATH = os.path.join(CHECKPOINTS_DIR, 'patchcore_dinov2_model.pt')
    SCORES_EXPORT_PATH = os.path.join(RESULTS_DIR, 'scores.npz')
    METRICS_EXPORT_PATH = os.path.join(RESULTS_DIR, 'evaluation_metrics.json')
    SUMMARY_EXPORT_PATH = os.path.join(RESULTS_DIR, 'summary.json')
    UMAP_PNG_PATH = os.path.join(PLOTS_DIR, 'umap_test_embeddings.png')
    UMAP_CSV_PATH = os.path.join(RESULTS_DIR, 'umap_test_embeddings.csv')
    for d in [CHECKPOINTS_DIR, PLOTS_DIR, RESULTS_DIR]:
        os.makedirs(d, exist_ok=True)
    print(f'Project root : {PROJECT_ROOT}')
    print(f'Backbone     : {BACKBONE_NAME}')
    print(f'Feature block: {VIT_FEATURE_BLOCK}  embed_dim={PATCH_EMBED_DIM}')
    print(f'Bank cap     : {MEMORY_BANK_MAX:,}  topk_ratio={TOPK_PATCH_RATIO}  k={PATCHCORE_NN_K}')
    print(f'Threshold    : q={THRESHOLD_QUANTILE}')
    print(f'FORCE_REBUILD_SCORES={FORCE_REBUILD_SCORES}  FORCE_RERUN_UMAP={FORCE_RERUN_UMAP}')
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "from pathlib import path\ncwd = path.cwd().resolve()\nproject_root = none\nfor candidate in [cwd, *cwd.parents]:\n    if (candidate / 'src' / 'wafer_defect').exists() and (candidate / 'configs').exists():\n        project_root = candidate\n        break\nif project_root is none:\n    raise runtimeerror('could not locate repo root containing src/wafer_defect and configs/')\nforce_rebuild_scores = false\nforce_rerun_umap = false\ndata_path = str(project_root / 'data' / 'raw' / 'lswmd.pkl')\nimage_size = 224\ntrain_normal_n = 40000\ntune_normal_n = 5000\ntest_normal_n = 5000\ntest_defect_n = 250\nbackbone_name = 'vit_base_patch14_dinov2.lvd142m'\nvit_feature_block = 9\npatch_embed_dim = 128\nmemory_bank_max = 600000\nscore_chunk = 512\npatchcore_nn_k = 3\ntopk_patch_ratio = 0.1\nbatch_size = 128\nnum_workers = 0\nthreshold_quantile = 0.95\nartifact_dir = str(project_root / 'experiments/anomaly_detection/patchcore/dinov2_vit_b14/x224/main/artifacts')\ncheckpoints_dir = os.path.join(artifact_dir, 'checkpoints')\nplots_dir = os.path.join(artifact_dir, 'plots')\nresults_dir = os.path.join(artifact_dir, 'results')\nmodel_export_path = os.path.join(checkpoints_dir, 'patchcore_dinov2_model.pt')\nscores_export_path = os.path.join(results_dir, 'scores.npz')\nmetrics_export_path = os.path.join(results_dir, 'evaluation_metrics.json')\nsummary_export_path = os.path.join(results_dir, 'summary.json')\numap_png_path = os.path.join(plots_dir, 'umap_test_embeddings.png')\numap_csv_path = os.path.join(results_dir, 'umap_test_embeddings.csv')\nfor d in [checkpoints_dir, plots_dir, results_dir]:\n    os.makedirs(d, exist_ok=true)\nprint(f'project root : {project_root}')\nprint(f'backbone     : {backbone_name}')\nprint(f'feature block: {vit_feature_block}  embed_dim={patch_embed_dim}')\nprint(f'bank cap     : {memory_bank_max:,}  topk_ratio={topk_patch_ratio}  k={patchcore_nn_k}')\nprint(f'threshold    : q={threshold_quantile}')\nprint(f'force_rebuild_scores={force_rebuild_scores}  force_rerun_umap={force_rerun_umap}')"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Load Dataset

In [ ]:
df = pd.read_pickle(DATA_PATH)
print('Raw shape:', df.shape)

def parse_failure_label(v):
    if v is None:
        return 'unknown'
    if isinstance(v, float) and np.isnan(v):
        return 'unknown'
    if isinstance(v, (list, tuple, np.ndarray)):
        a = np.array(v).reshape(-1)
        return 'unknown' if len(a) == 0 else str(a[0])
    return str(v)
df = df.copy()
df['failure_label'] = df['failureType'].apply(parse_failure_label).str.strip()
invalid = {'0', 'unknown', 'nan', 'None', '[]'}
df = df[~df['failure_label'].isin(invalid)].copy()
df['is_anomaly'] = (df['failure_label'].str.lower() != 'none').astype(int)
normal_df = df[df['is_anomaly'] == 0].copy()
defect_df = df[df['is_anomaly'] == 1].copy()
print(f'Labeled: {len(df):,}   Normal: {len(normal_df):,}   Defect: {len(defect_df):,}')
print('\nDefect breakdown:')
print(defect_df['failure_label'].value_counts())

## Split

In [ ]:
rng = np.random.default_rng(SEED)
ns = normal_df.iloc[rng.permutation(len(normal_df))].reset_index(drop=True)
ds = defect_df.iloc[rng.permutation(len(defect_df))].reset_index(drop=True)
a = TRAIN_NORMAL_N
b = a + TUNE_NORMAL_N
c = b + TEST_NORMAL_N
train_normal_df = ns.iloc[0:a].copy()
tune_normal_df = ns.iloc[a:b].copy()
test_normal_df = ns.iloc[b:c].copy()
test_defect_df = ds.iloc[0:TEST_DEFECT_N].copy()
del df, normal_df, defect_df, ns, ds
gc.collect()
print(f'Train normal : {len(train_normal_df):>7,}  (memory bank)')
print(f'Tune  normal : {len(tune_normal_df):>7,}  (threshold calibration)')
print(f'Test  normal : {len(test_normal_df):>7,}')
print(f'Test  defect : {len(test_defect_df):>7,}')
print('\nDefect classes in test set:')
print(test_defect_df['failure_label'].value_counts())

## Dataset

Lazy dataset: raw numpy wafer maps stored in the DataFrame, converted to tensors per batch only.

In [ ]:
class WaferDataset(Dataset):

    def __init__(self, frame: pd.DataFrame, size: int=224):
        self.maps = frame['waferMap'].values
        self.labels = frame['is_anomaly'].values.astype(np.int64)
        self.size = size

    def __len__(self):
        return len(self.maps)

    def __getitem__(self, idx):
        arr = np.clip(np.array(self.maps[idx], dtype=np.int64), 0, 2)
        x = torch.tensor(arr, dtype=torch.long)
        x = F.one_hot(x, num_classes=3).permute(2, 0, 1).float()
        x = F.interpolate(x.unsqueeze(0), size=(self.size, self.size), mode='nearest').squeeze(0)
        return (x, int(self.labels[idx]))
loader_kw = dict(batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=USE_CUDA, persistent_workers=NUM_WORKERS > 0)
train_loader = DataLoader(WaferDataset(train_normal_df, IMAGE_SIZE), **loader_kw)
tune_normal_loader = DataLoader(WaferDataset(tune_normal_df, IMAGE_SIZE), **loader_kw)
test_normal_loader = DataLoader(WaferDataset(test_normal_df, IMAGE_SIZE), **loader_kw)
test_defect_loader = DataLoader(WaferDataset(test_defect_df, IMAGE_SIZE), **loader_kw)
xb, yb = next(iter(train_loader))
print(f'Batch shape: {tuple(xb.shape)}  dtype={xb.dtype}')

## Define Model

DINOv2 ViT-B/14 loaded with pretrained weights. Fully frozen. A forward hook captures block `VIT_FEATURE_BLOCK` patch tokens, which are projected to `PATCH_EMBED_DIM` dimensions for the memory bank.

**Why block 9?** DINOv2's self-distillation training preserves local spatial structure through deeper layers than supervised ViT. Block 9 retains strong patch-level features while incorporating more semantic context than block 6. Block 11 (final) can also be tried - unlike supervised ViT where it collapses, DINOv2 block 11 still retains useful local geometry.

In [ ]:
class DINOv2PatchExtractor(nn.Module):
    """
    Frozen DINOv2 ViT-B/14 backbone.
    Hooks block[VIT_FEATURE_BLOCK] to capture [B, N+1, 768] token output.
    Drops CLS token -> [B, 256, 768] spatial patch tokens.
    Projects to [B, 256, PATCH_EMBED_DIM].
    """

    def __init__(self, block_idx: int=VIT_FEATURE_BLOCK, proj_dim: int=PATCH_EMBED_DIM):
        super().__init__()
        self.vit = timm.create_model(BACKBONE_NAME, pretrained=True, num_classes=0)
        self._feat = None
        self.vit.blocks[block_idx].register_forward_hook(lambda m, i, o: setattr(self, '_feat', o))
        embed_dim = self.vit.embed_dim
        self.proj = nn.Linear(embed_dim, proj_dim, bias=False)

    def forward(self, x):
        self.vit(x)
        return self._feat[:, 1:, :]
extractor = DINOv2PatchExtractor().to(DEVICE).eval()
for p in extractor.parameters():
    p.requires_grad = False
with torch.inference_mode():
    dummy = torch.zeros(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
    out = extractor(dummy)
    proj = extractor.proj(out)
n_tokens = out.shape[1]
print(f'Backbone     : {BACKBONE_NAME}')
print(f'Block-{VIT_FEATURE_BLOCK} output : {tuple(out.shape)}  ({n_tokens} tokens/image)')
print(f'After proj   : {tuple(proj.shape)}')
print(f'Tokens vs ViT-B/16: {n_tokens} vs 196 ({n_tokens - 196:+d})')

## Train and Score

Builds the PatchCore memory bank (one forward pass over training normals) and scores all splits. When `FORCE_REBUILD_SCORES=False` and `scores.npz` exists, loads saved scores and skips the memory bank build entirely.

In [ ]:
try:
    REQUIRED_KEYS = {'tune_normal_scores', 'test_normal_scores', 'test_defect_scores', 'threshold'}
    saved_keys = set()
    if os.path.exists(SCORES_EXPORT_PATH):
        with np.load(SCORES_EXPORT_PATH) as _f:
            saved_keys = set(_f.files)
    REUSE_SCORES = REQUIRED_KEYS.issubset(saved_keys) and (not FORCE_REBUILD_SCORES)
    memory_bank = None
    if REUSE_SCORES:
        with np.load(SCORES_EXPORT_PATH) as d:
            tune_normal_scores = d['tune_normal_scores']
            test_normal_scores = d['test_normal_scores']
            test_defect_scores = d['test_defect_scores']
            best_thresh = float(d['threshold'])
        print(f'Loaded saved scores from: {SCORES_EXPORT_PATH}')
        print(f'Threshold: {best_thresh:.6f}')
    else:

        def extract_embeddings(xb: torch.Tensor) -> torch.Tensor:
            """L2-normalised patch embeddings: [B*N_tokens, proj_dim] on GPU."""
            with torch.inference_mode():
                with torch.amp.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_CUDA):
                    feat = extractor(xb)
                    emb = extractor.proj(feat)
                emb = emb.float().reshape(-1, emb.shape[-1])
                emb = F.normalize(emb, p=2, dim=1)
            return emb
        sampled, total_seen, sample_ratio = ([], 0, None)
        print('Building memory bank...')
        bank_iter = tqdm(enumerate(train_loader), total=len(train_loader), desc='Bank build', unit='batch')
        for step, (xb, _) in bank_iter:
            xb = xb.to(DEVICE)
            emb = extract_embeddings(xb)
            total_seen += len(emb)
            if sample_ratio is None:
                tokens_per_img = len(emb) // len(xb)
                estimated_total = tokens_per_img * len(train_normal_df)
                sample_ratio = min(1.0, MEMORY_BANK_MAX / estimated_total)
                print(f'  Tokens/image : {tokens_per_img}  (DINOv2 patch14 at {IMAGE_SIZE}px)')
                print(f'  Est. total   : {estimated_total:,}')
                print(f'  Sample ratio : {sample_ratio:.5f}')
            if sample_ratio < 1.0:
                k = max(1, int(round(len(emb) * sample_ratio)))
                idx = torch.randperm(len(emb), device=DEVICE)[:k]
                emb = emb[idx]
            sampled.append(emb)
            if (step + 1) % 20 == 0 or step + 1 == len(train_loader):
                bank_iter.set_postfix(bank_tokens=f'{sum((len(e) for e in sampled)):,}')
        memory_bank = torch.cat(sampled, dim=0)
        del sampled
        gc.collect()
        if len(memory_bank) > MEMORY_BANK_MAX:
            idx = torch.randperm(len(memory_bank), device=DEVICE)[:MEMORY_BANK_MAX]
            memory_bank = memory_bank[idx]
        vram_mb = memory_bank.element_size() * memory_bank.nelement() / 1000000.0
        print(f'Final bank : {len(memory_bank):,} x {memory_bank.shape[1]}-d  ({vram_mb:.1f} MB VRAM)')

        @torch.inference_mode()
        def score_loader(loader: DataLoader, desc: str='Scoring') -> np.ndarray:
            scores = []
            for xb, _ in tqdm(loader, desc=desc, leave=False):
                xb = xb.to(DEVICE)
                emb = extract_embeddings(xb)
                B = len(xb)
                P = emb.shape[0] // B
                patch_dists = torch.empty(B * P, dtype=torch.float32, device=DEVICE)
                for start in range(0, B * P, SCORE_CHUNK):
                    end = min(start + SCORE_CHUNK, B * P)
                    chunk = emb[start:end]
                    dists = torch.cdist(chunk, memory_bank)
                    knn = dists.topk(PATCHCORE_NN_K, dim=1, largest=False).values
                    patch_dists[start:end] = knn.mean(dim=1)
                patch_dists = patch_dists.view(B, P)
                topk = max(1, int(P * TOPK_PATCH_RATIO))
                img_scores = patch_dists.topk(topk, dim=1).values.mean(dim=1)
                scores.extend(img_scores.cpu().tolist())
            return np.array(scores)
        print('Scoring splits...')
        tune_normal_scores = score_loader(tune_normal_loader, 'Score tune-normal')
        test_normal_scores = score_loader(test_normal_loader, 'Score test-normal')
        test_defect_scores = score_loader(test_defect_loader, 'Score test-defect')
        best_thresh = float(np.quantile(tune_normal_scores, THRESHOLD_QUANTILE))
        print(f'Threshold (q={THRESHOLD_QUANTILE:.2f} on tune-normal): {best_thresh:.6f}')
        np.savez_compressed(SCORES_EXPORT_PATH, tune_normal_scores=tune_normal_scores, test_normal_scores=test_normal_scores, test_defect_scores=test_defect_scores, threshold=np.array(best_thresh))
        print(f'Scores saved -> {SCORES_EXPORT_PATH}')
    print(f'tune_normal: mean={tune_normal_scores.mean():.4f}  std={tune_normal_scores.std():.4f}')
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = 'required_keys = {\'tune_normal_scores\', \'test_normal_scores\', \'test_defect_scores\', \'threshold\'}\nsaved_keys = set()\nif os.path.exists(scores_export_path):\n    with np.load(scores_export_path) as _f:\n        saved_keys = set(_f.files)\nreuse_scores = required_keys.issubset(saved_keys) and (not force_rebuild_scores)\nmemory_bank = none\nif reuse_scores:\n    with np.load(scores_export_path) as d:\n        tune_normal_scores = d[\'tune_normal_scores\']\n        test_normal_scores = d[\'test_normal_scores\']\n        test_defect_scores = d[\'test_defect_scores\']\n        best_thresh = float(d[\'threshold\'])\n    print(f\'loaded saved scores from: {scores_export_path}\')\n    print(f\'threshold: {best_thresh:.6f}\')\nelse:\n\n    def extract_embeddings(xb: torch.tensor) -> torch.tensor:\n        """l2-normalised patch embeddings: [b*n_tokens, proj_dim] on gpu."""\n        with torch.inference_mode():\n            with torch.amp.autocast(device_type=\'cuda\', dtype=torch.float16, enabled=use_cuda):\n                feat = extractor(xb)\n                emb = extractor.proj(feat)\n            emb = emb.float().reshape(-1, emb.shape[-1])\n            emb = f.normalize(emb, p=2, dim=1)\n        return emb\n    sampled, total_seen, sample_ratio = ([], 0, none)\n    print(\'building memory bank...\')\n    bank_iter = tqdm(enumerate(train_loader), total=len(train_loader), desc=\'bank build\', unit=\'batch\')\n    for step, (xb, _) in bank_iter:\n        xb = xb.to(device)\n        emb = extract_embeddings(xb)\n        total_seen += len(emb)\n        if sample_ratio is none:\n            tokens_per_img = len(emb) // len(xb)\n            estimated_total = tokens_per_img * len(train_normal_df)\n            sample_ratio = min(1.0, memory_bank_max / estimated_total)\n            print(f\'  tokens/image : {tokens_per_img}  (dinov2 patch14 at {image_size}px)\')\n            print(f\'  est. total   : {estimated_total:,}\')\n            print(f\'  sample ratio : {sample_ratio:.5f}\')\n        if sample_ratio < 1.0:\n            k = max(1, int(round(len(emb) * sample_ratio)))\n            idx = torch.randperm(len(emb), device=device)[:k]\n            emb = emb[idx]\n        sampled.append(emb)\n        if (step + 1) % 20 == 0 or step + 1 == len(train_loader):\n            bank_iter.set_postfix(bank_tokens=f\'{sum((len(e) for e in sampled)):,}\')\n    memory_bank = torch.cat(sampled, dim=0)\n    del sampled\n    gc.collect()\n    if len(memory_bank) > memory_bank_max:\n        idx = torch.randperm(len(memory_bank), device=device)[:memory_bank_max]\n        memory_bank = memory_bank[idx]\n    vram_mb = memory_bank.element_size() * memory_bank.nelement() / 1000000.0\n    print(f\'final bank : {len(memory_bank):,} x {memory_bank.shape[1]}-d  ({vram_mb:.1f} mb vram)\')\n\n    @torch.inference_mode()\n    def score_loader(loader: dataloader, desc: str=\'scoring\') -> np.ndarray:\n        scores = []\n        for xb, _ in tqdm(loader, desc=desc, leave=false):\n            xb = xb.to(device)\n            emb = extract_embeddings(xb)\n            b = len(xb)\n            p = emb.shape[0] // b\n            patch_dists = torch.empty(b * p, dtype=torch.float32, device=device)\n            for start in range(0, b * p, score_chunk):\n                end = min(start + score_chunk, b * p)\n                chunk = emb[start:end]\n                dists = torch.cdist(chunk, memory_bank)\n                knn = dists.topk(patchcore_nn_k, dim=1, largest=false).values\n                patch_dists[start:end] = knn.mean(dim=1)\n            patch_dists = patch_dists.view(b, p)\n            topk = max(1, int(p * topk_patch_ratio))\n            img_scores = patch_dists.topk(topk, dim=1).values.mean(dim=1)\n            scores.extend(img_scores.cpu().tolist())\n        return np.array(scores)\n    print(\'scoring splits...\')\n    tune_normal_scores = score_loader(tune_normal_loader, \'score tune-normal\')\n    test_normal_scores = score_loader(test_normal_loader, \'score test-normal\')\n    test_defect_scores = score_loader(test_defect_loader, \'score test-defect\')\n    best_thresh = floa'
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Threshold Calibration

95th percentile of tune-normal scores. No defect labels used.

In [ ]:
try:
    best_fpr = float((tune_normal_scores > best_thresh).mean())
    print(f'Threshold : {best_thresh:.6f}  (q={THRESHOLD_QUANTILE:.2f})')
    print(f'FPR on tune-normal: {best_fpr:.4f}')
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(tune_normal_scores, bins=50, alpha=0.7, color='steelblue', label='Tune normal')
    ax.axvline(best_thresh, color='red', linestyle='--', lw=2, label=f'Threshold={best_thresh:.4f} (q={THRESHOLD_QUANTILE:.2f})')
    ax.set_title('Tune-Normal Score Distribution (DINOv2 ViT-B/14)')
    ax.set_xlabel('Anomaly Score')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'threshold_selection.png'), dpi=140, bbox_inches='tight')
    plt.show()
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "best_fpr = float((tune_normal_scores > best_thresh).mean())\nprint(f'threshold : {best_thresh:.6f}  (q={threshold_quantile:.2f})')\nprint(f'fpr on tune-normal: {best_fpr:.4f}')\nfig, ax = plt.subplots(figsize=(8, 4))\nax.hist(tune_normal_scores, bins=50, alpha=0.7, color='steelblue', label='tune normal')\nax.axvline(best_thresh, color='red', linestyle='--', lw=2, label=f'threshold={best_thresh:.4f} (q={threshold_quantile:.2f})')\nax.set_title('tune-normal score distribution (dinov2 vit-b/14)')\nax.set_xlabel('anomaly score')\nax.set_ylabel('count')\nax.legend(fontsize=9)\nplt.tight_layout()\nplt.savefig(os.path.join(plots_dir, 'threshold_selection.png'), dpi=140, bbox_inches='tight')\nplt.show()"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Evaluate

Overall and per-class metrics at the calibrated threshold.

In [ ]:
try:
    all_scores = np.concatenate([test_normal_scores, test_defect_scores])
    all_labels = np.concatenate([np.zeros(len(test_normal_scores), dtype=int), np.ones(len(test_defect_scores), dtype=int)])
    predictions = (all_scores > best_thresh).astype(int)
    roc_auc = roc_auc_score(all_labels, all_scores)
    auprc = average_precision_score(all_labels, all_scores)
    fpr_arr, tpr_arr, _ = roc_curve(all_labels, all_scores)
    precision_arr, recall_arr, _ = precision_recall_curve(all_labels, all_scores)
    f1_val = f1_score(all_labels, predictions, pos_label=1, zero_division=0)
    precision_val = precision_score(all_labels, predictions, pos_label=1, zero_division=0)
    recall_val = recall_score(all_labels, predictions, pos_label=1, zero_division=0)
    print('-- Overall Test Results --------------------------')
    print(f'ROC-AUC  : {roc_auc:.4f}')
    print(f'AUPRC    : {auprc:.4f}')
    print(f'Threshold: {best_thresh:.6f}  (q={THRESHOLD_QUANTILE:.2f})')
    print()
    print(classification_report(all_labels, predictions, target_names=['Normal', 'Defect'], digits=4))
    defect_class_labels = test_defect_df['failure_label'].values
    defect_preds = (test_defect_scores > best_thresh).astype(int)
    print('-- Per-class defect recall ---------------------------------------')
    print(f"  {'Defect type':<14}  {'N':>5}  {'Detected':>8}  {'Recall':>7}  {'Mean score':>10}")
    print('  ' + '-' * 52)
    perclass_results = {}
    for cls in sorted(np.unique(defect_class_labels)):
        mask = defect_class_labels == cls
        n = mask.sum()
        detected = defect_preds[mask].sum()
        recall = detected / n
        mean_score = test_defect_scores[mask].mean()
        perclass_results[cls] = {'n': int(n), 'detected': int(detected), 'recall': float(recall), 'mean_score': float(mean_score)}
        print(f'  {cls:<14}  {n:>5}  {detected:>8}  {recall:>6.1%}  {mean_score:>10.4f}')
    overall_defect_recall = defect_preds.sum() / len(defect_preds)
    print('  ' + '-' * 52)
    print(f"  {'ALL DEFECTS':<14}  {len(defect_preds):>5}  {defect_preds.sum():>8}  {overall_defect_recall:>6.1%}")
    print('\n-- vs Frozen ViT-B/16 baseline -------------------')
    baseline = {'F1': 0.595, 'AUROC': 0.956, 'AUPRC': 0.671}
    print(f"  F1    : {f1_val:.4f}  (baseline {baseline['F1']:.3f}, delta {f1_val - baseline['F1']:+.3f})")
    print(f"  AUROC : {roc_auc:.4f}  (baseline {baseline['AUROC']:.3f}, delta {roc_auc - baseline['AUROC']:+.3f})")
    print(f"  AUPRC : {auprc:.4f}  (baseline {baseline['AUPRC']:.3f}, delta {auprc - baseline['AUPRC']:+.3f})")
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = 'all_scores = np.concatenate([test_normal_scores, test_defect_scores])\nall_labels = np.concatenate([np.zeros(len(test_normal_scores), dtype=int), np.ones(len(test_defect_scores), dtype=int)])\npredictions = (all_scores > best_thresh).astype(int)\nroc_auc = roc_auc_score(all_labels, all_scores)\nauprc = average_precision_score(all_labels, all_scores)\nfpr_arr, tpr_arr, _ = roc_curve(all_labels, all_scores)\nprecision_arr, recall_arr, _ = precision_recall_curve(all_labels, all_scores)\nf1_val = f1_score(all_labels, predictions, pos_label=1, zero_division=0)\nprecision_val = precision_score(all_labels, predictions, pos_label=1, zero_division=0)\nrecall_val = recall_score(all_labels, predictions, pos_label=1, zero_division=0)\nprint(\'-- overall test results --------------------------\')\nprint(f\'roc-auc  : {roc_auc:.4f}\')\nprint(f\'auprc    : {auprc:.4f}\')\nprint(f\'threshold: {best_thresh:.6f}  (q={threshold_quantile:.2f})\')\nprint()\nprint(classification_report(all_labels, predictions, target_names=[\'normal\', \'defect\'], digits=4))\ndefect_class_labels = test_defect_df[\'failure_label\'].values\ndefect_preds = (test_defect_scores > best_thresh).astype(int)\nprint(\'-- per-class defect recall ---------------------------------------\')\nprint(f"  {\'defect type\':<14}  {\'n\':>5}  {\'detected\':>8}  {\'recall\':>7}  {\'mean score\':>10}")\nprint(\'  \' + \'-\' * 52)\nperclass_results = {}\nfor cls in sorted(np.unique(defect_class_labels)):\n    mask = defect_class_labels == cls\n    n = mask.sum()\n    detected = defect_preds[mask].sum()\n    recall = detected / n\n    mean_score = test_defect_scores[mask].mean()\n    perclass_results[cls] = {\'n\': int(n), \'detected\': int(detected), \'recall\': float(recall), \'mean_score\': float(mean_score)}\n    print(f\'  {cls:<14}  {n:>5}  {detected:>8}  {recall:>6.1%}  {mean_score:>10.4f}\')\noverall_defect_recall = defect_preds.sum() / len(defect_preds)\nprint(\'  \' + \'-\' * 52)\nprint(f"  {\'all defects\':<14}  {len(defect_preds):>5}  {defect_preds.sum():>8}  {overall_defect_recall:>6.1%}")\nprint(\'\\n-- vs frozen vit-b/16 baseline -------------------\')\nbaseline = {\'f1\': 0.595, \'auroc\': 0.956, \'auprc\': 0.671}\nprint(f"  f1    : {f1_val:.4f}  (baseline {baseline[\'f1\']:.3f}, delta {f1_val - baseline[\'f1\']:+.3f})")\nprint(f"  auroc : {roc_auc:.4f}  (baseline {baseline[\'auroc\']:.3f}, delta {roc_auc - baseline[\'auroc\']:+.3f})")\nprint(f"  auprc : {auprc:.4f}  (baseline {baseline[\'auprc\']:.3f}, delta {auprc - baseline[\'auprc\']:+.3f})")'
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


In [ ]:
try:
    fig = plt.figure(figsize=(21, 10))
    gs = fig.add_gridspec(2, 4, hspace=0.4, wspace=0.35)
    ax_dist = fig.add_subplot(gs[0, 0])
    ax_roc = fig.add_subplot(gs[0, 1])
    ax_pr = fig.add_subplot(gs[0, 2])
    ax_cm = fig.add_subplot(gs[0, 3])
    ax_pc = fig.add_subplot(gs[1, :])
    ax_dist.hist(test_normal_scores, bins=50, alpha=0.7, color='steelblue', label='Normal')
    ax_dist.hist(test_defect_scores, bins=50, alpha=0.7, color='tomato', label='Defect')
    ax_dist.axvline(best_thresh, color='black', linestyle='--', label=f'Threshold={best_thresh:.4f}')
    ax_dist.set_title('Score Distributions')
    ax_dist.set_xlabel('Anomaly Score')
    ax_dist.set_ylabel('Count')
    ax_dist.legend(fontsize=9)
    ax_roc.plot(fpr_arr, tpr_arr, color='darkorange', lw=2, label=f'ROC-AUC={roc_auc:.4f}')
    ax_roc.plot([0, 1], [0, 1], 'k--', alpha=0.4)
    op_fpr = (test_normal_scores > best_thresh).mean()
    op_tpr = (test_defect_scores > best_thresh).mean()
    ax_roc.scatter([op_fpr], [op_tpr], color='red', zorder=5, label=f'FPR={op_fpr:.3f} TPR={op_tpr:.3f}')
    ax_roc.set_title('ROC Curve')
    ax_roc.set_xlabel('FPR')
    ax_roc.set_ylabel('TPR')
    ax_roc.legend(fontsize=9)
    baseline_pr = all_labels.mean()
    ax_pr.plot(recall_arr, precision_arr, color='purple', lw=2, label=f'AUPRC={auprc:.4f}')
    ax_pr.axhline(baseline_pr, color='gray', linestyle='--', lw=1, label=f'No-skill={baseline_pr:.3f}')
    ax_pr.set_title('Precision-Recall')
    ax_pr.set_xlabel('Recall')
    ax_pr.set_ylabel('Precision')
    ax_pr.set_xlim(0, 1)
    ax_pr.set_ylim(0, 1.02)
    ax_pr.legend(fontsize=9)
    cm = confusion_matrix(all_labels, predictions)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Pred Normal', 'Pred Defect'], yticklabels=['True Normal', 'True Defect'], ax=ax_cm)
    ax_cm.set_title('Confusion Matrix')
    classes = list(perclass_results.keys())
    recalls = [perclass_results[c]['recall'] for c in classes]
    counts = [perclass_results[c]['n'] for c in classes]
    mean_scores = [perclass_results[c]['mean_score'] for c in classes]
    order = np.argsort(recalls)
    classes = [classes[i] for i in order]
    recalls = [recalls[i] for i in order]
    counts = [counts[i] for i in order]
    mean_scores = [mean_scores[i] for i in order]
    bar_colors = ['#e05c5c' if r < 0.5 else '#f5a623' if r < 0.8 else '#4caf50' for r in recalls]
    bars = ax_pc.bar(classes, recalls, color=bar_colors, edgecolor='white', width=0.6)
    ax_pc.axhline(overall_defect_recall, color='navy', linestyle='--', lw=1.5, label=f'Overall recall={overall_defect_recall:.1%}')
    ax_pc.axhline(0.8, color='gray', linestyle=':', lw=1, label='80% target')
    ax_pc.set_ylim(0, 1.12)
    ax_pc.set_ylabel('Detection Recall')
    ax_pc.set_title('Per-Class Defect Recall  (red<50% | orange 50-80% | green>=80%)')
    ax_pc.legend(fontsize=9)
    for bar, rec, n in zip(bars, recalls, counts):
        ax_pc.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02, f'{rec:.0%}\n(n={n})', ha='center', va='bottom', fontsize=9, fontweight='bold')
    plt.suptitle(f'PatchCore DINOv2 ViT-B/14 block-{VIT_FEATURE_BLOCK}  |  ROC-AUC={roc_auc:.4f}  |  AUPRC={auprc:.4f}  |  F1={f1_val:.4f}', fontsize=13, fontweight='bold')
    plt.savefig(os.path.join(PLOTS_DIR, 'evaluation_results.png'), dpi=130, bbox_inches='tight')
    plt.show()
    defect_recall_df = pd.DataFrame([{'failure_label': c, **v} for c, v in perclass_results.items()]).sort_values('recall').reset_index(drop=True)
    defect_recall_df.to_csv(os.path.join(RESULTS_DIR, 'defect_recall.csv'), index=False)
    display(defect_recall_df)
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "fig = plt.figure(figsize=(21, 10))\ngs = fig.add_gridspec(2, 4, hspace=0.4, wspace=0.35)\nax_dist = fig.add_subplot(gs[0, 0])\nax_roc = fig.add_subplot(gs[0, 1])\nax_pr = fig.add_subplot(gs[0, 2])\nax_cm = fig.add_subplot(gs[0, 3])\nax_pc = fig.add_subplot(gs[1, :])\nax_dist.hist(test_normal_scores, bins=50, alpha=0.7, color='steelblue', label='normal')\nax_dist.hist(test_defect_scores, bins=50, alpha=0.7, color='tomato', label='defect')\nax_dist.axvline(best_thresh, color='black', linestyle='--', label=f'threshold={best_thresh:.4f}')\nax_dist.set_title('score distributions')\nax_dist.set_xlabel('anomaly score')\nax_dist.set_ylabel('count')\nax_dist.legend(fontsize=9)\nax_roc.plot(fpr_arr, tpr_arr, color='darkorange', lw=2, label=f'roc-auc={roc_auc:.4f}')\nax_roc.plot([0, 1], [0, 1], 'k--', alpha=0.4)\nop_fpr = (test_normal_scores > best_thresh).mean()\nop_tpr = (test_defect_scores > best_thresh).mean()\nax_roc.scatter([op_fpr], [op_tpr], color='red', zorder=5, label=f'fpr={op_fpr:.3f} tpr={op_tpr:.3f}')\nax_roc.set_title('roc curve')\nax_roc.set_xlabel('fpr')\nax_roc.set_ylabel('tpr')\nax_roc.legend(fontsize=9)\nbaseline_pr = all_labels.mean()\nax_pr.plot(recall_arr, precision_arr, color='purple', lw=2, label=f'auprc={auprc:.4f}')\nax_pr.axhline(baseline_pr, color='gray', linestyle='--', lw=1, label=f'no-skill={baseline_pr:.3f}')\nax_pr.set_title('precision-recall')\nax_pr.set_xlabel('recall')\nax_pr.set_ylabel('precision')\nax_pr.set_xlim(0, 1)\nax_pr.set_ylim(0, 1.02)\nax_pr.legend(fontsize=9)\ncm = confusion_matrix(all_labels, predictions)\nsns.heatmap(cm, annot=true, fmt='d', cmap='blues', xticklabels=['pred normal', 'pred defect'], yticklabels=['true normal', 'true defect'], ax=ax_cm)\nax_cm.set_title('confusion matrix')\nclasses = list(perclass_results.keys())\nrecalls = [perclass_results[c]['recall'] for c in classes]\ncounts = [perclass_results[c]['n'] for c in classes]\nmean_scores = [perclass_results[c]['mean_score'] for c in classes]\norder = np.argsort(recalls)\nclasses = [classes[i] for i in order]\nrecalls = [recalls[i] for i in order]\ncounts = [counts[i] for i in order]\nmean_scores = [mean_scores[i] for i in order]\nbar_colors = ['#e05c5c' if r < 0.5 else '#f5a623' if r < 0.8 else '#4caf50' for r in recalls]\nbars = ax_pc.bar(classes, recalls, color=bar_colors, edgecolor='white', width=0.6)\nax_pc.axhline(overall_defect_recall, color='navy', linestyle='--', lw=1.5, label=f'overall recall={overall_defect_recall:.1%}')\nax_pc.axhline(0.8, color='gray', linestyle=':', lw=1, label='80% target')\nax_pc.set_ylim(0, 1.12)\nax_pc.set_ylabel('detection recall')\nax_pc.set_title('per-class defect recall  (red<50% | orange 50-80% | green>=80%)')\nax_pc.legend(fontsize=9)\nfor bar, rec, n in zip(bars, recalls, counts):\n    ax_pc.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02, f'{rec:.0%}\\n(n={n})', ha='center', va='bottom', fontsize=9, fontweight='bold')\nplt.suptitle(f'patchcore dinov2 vit-b/14 block-{vit_feature_block}  |  roc-auc={roc_auc:.4f}  |  auprc={auprc:.4f}  |  f1={f1_val:.4f}', fontsize=13, fontweight='bold')\nplt.savefig(os.path.join(plots_dir, 'evaluation_results.png'), dpi=130, bbox_inches='tight')\nplt.show()\ndefect_recall_df = pd.dataframe([{'failure_label': c, **v} for c, v in perclass_results.items()]).sort_values('recall').reset_index(drop=true)\ndefect_recall_df.to_csv(os.path.join(results_dir, 'defect_recall.csv'), index=false)\ndisplay(defect_recall_df)"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## UMAP Visualization

Projects mean-pooled patch embeddings into 2D. Displays saved PNG when `FORCE_RERUN_UMAP=False`.

In [ ]:
try:
    from sklearn.decomposition import PCA
    if not FORCE_RERUN_UMAP and os.path.exists(UMAP_PNG_PATH):
        print(f'Displaying saved UMAP: {UMAP_PNG_PATH}')
        display(IPImage(filename=UMAP_PNG_PATH))
    elif FORCE_RERUN_UMAP:
        if FORCE_RERUN_UMAP:
            if FORCE_RERUN_UMAP:
                if FORCE_RERUN_UMAP:
                    try:
                        import umap.umap_ as umap
                    except ImportError:
                        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'umap-learn', '-q'])
                        import umap.umap_ as umap
                    MAX_UMAP = 2000

                    def collect_image_embeddings(loader, max_images=2000, desc='embed'):
                        embs, labels = ([], [])
                        seen = 0
                        with torch.inference_mode():
                            for xb, yb in tqdm(loader, desc=desc, unit='batch'):
                                xb = xb.to(DEVICE)
                                with torch.amp.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_CUDA):
                                    feat = extractor(xb)
                                    emb = extractor.proj(feat)
                                img_emb = F.normalize(emb.float().mean(dim=1), p=2, dim=1)
                                embs.append(img_emb.cpu().numpy())
                                labels.append(yb.cpu().numpy())
                                seen += len(yb)
                                if seen >= max_images:
                                    break
                        return (np.concatenate(embs)[:max_images], np.concatenate(labels)[:max_images])
                    print('Collecting embeddings for UMAP...')
                    Xn, yn = collect_image_embeddings(test_normal_loader, MAX_UMAP, 'Embed test-normal')
                    Xd, yd = collect_image_embeddings(test_defect_loader, MAX_UMAP, 'Embed test-defect')
                    X = np.concatenate([Xn, Xd])
                    y = np.concatenate([yn, yd]).astype(int)
                    n_pca = min(32, X.shape[1], X.shape[0] - 1)
                    Xp = PCA(n_components=n_pca, random_state=SEED).fit_transform(X)
                    reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, metric='cosine', random_state=SEED, transform_seed=SEED, low_memory=True, verbose=True)
                    coords = reducer.fit_transform(Xp)
                    fig, ax = plt.subplots(figsize=(8, 6))
                    m0, m1 = (y == 0, y == 1)
                    ax.scatter(coords[m0, 0], coords[m0, 1], s=8, alpha=0.45, c='steelblue', label='Normal', linewidths=0)
                    ax.scatter(coords[m1, 0], coords[m1, 1], s=8, alpha=0.6, c='tomato', label='Defect', linewidths=0)
                    ax.set_title(f'UMAP - DINOv2 ViT-B/14 PatchCore (block {VIT_FEATURE_BLOCK})')
                    ax.set_xlabel('UMAP-1')
                    ax.set_ylabel('UMAP-2')
                    ax.legend()
                    plt.tight_layout()
                    fig.savefig(UMAP_PNG_PATH, dpi=160, bbox_inches='tight')
                    plt.show()
                    pd.DataFrame({'umap_1': coords[:, 0], 'umap_2': coords[:, 1], 'label': y}).to_csv(UMAP_CSV_PATH, index=False)
                    print(f'Saved: {UMAP_PNG_PATH}')
                else:
                    print('[WARNING] The rerun/training flags are False and the saved artifacts for this section are missing. Skipping this section.')
            else:
                print('[WARNING] The rerun/training flags are False and the saved artifacts for this section are missing. Skipping this section.')
        else:
            print('[WARNING] The rerun/training flags are False and the saved artifacts for this section are missing. Skipping this section.')
    else:
        print('[WARNING] The rerun/training flags are False and the saved artifacts for this section are missing. Skipping this section.')
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "from sklearn.decomposition import pca\nif not force_rerun_umap and os.path.exists(umap_png_path):\n    print(f'displaying saved umap: {umap_png_path}')\n    display(ipimage(filename=umap_png_path))\nelif force_rerun_umap:\n    try:\n        import umap.umap_ as umap\n    except importerror:\n        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'umap-learn', '-q'])\n        import umap.umap_ as umap\n    max_umap = 2000\n\n    def collect_image_embeddings(loader, max_images=2000, desc='embed'):\n        embs, labels = ([], [])\n        seen = 0\n        with torch.inference_mode():\n            for xb, yb in tqdm(loader, desc=desc, unit='batch'):\n                xb = xb.to(device)\n                with torch.amp.autocast(device_type='cuda', dtype=torch.float16, enabled=use_cuda):\n                    feat = extractor(xb)\n                    emb = extractor.proj(feat)\n                img_emb = f.normalize(emb.float().mean(dim=1), p=2, dim=1)\n                embs.append(img_emb.cpu().numpy())\n                labels.append(yb.cpu().numpy())\n                seen += len(yb)\n                if seen >= max_images:\n                    break\n        return (np.concatenate(embs)[:max_images], np.concatenate(labels)[:max_images])\n    print('collecting embeddings for umap...')\n    xn, yn = collect_image_embeddings(test_normal_loader, max_umap, 'embed test-normal')\n    xd, yd = collect_image_embeddings(test_defect_loader, max_umap, 'embed test-defect')\n    x = np.concatenate([xn, xd])\n    y = np.concatenate([yn, yd]).astype(int)\n    n_pca = min(32, x.shape[1], x.shape[0] - 1)\n    xp = pca(n_components=n_pca, random_state=seed).fit_transform(x)\n    reducer = umap.umap(n_components=2, n_neighbors=15, min_dist=0.1, metric='cosine', random_state=seed, transform_seed=seed, low_memory=true, verbose=true)\n    coords = reducer.fit_transform(xp)\n    fig, ax = plt.subplots(figsize=(8, 6))\n    m0, m1 = (y == 0, y == 1)\n    ax.scatter(coords[m0, 0], coords[m0, 1], s=8, alpha=0.45, c='steelblue', label='normal', linewidths=0)\n    ax.scatter(coords[m1, 0], coords[m1, 1], s=8, alpha=0.6, c='tomato', label='defect', linewidths=0)\n    ax.set_title(f'umap - dinov2 vit-b/14 patchcore (block {vit_feature_block})')\n    ax.set_xlabel('umap-1')\n    ax.set_ylabel('umap-2')\n    ax.legend()\n    plt.tight_layout()\n    fig.savefig(umap_png_path, dpi=160, bbox_inches='tight')\n    plt.show()\n    pd.dataframe({'umap_1': coords[:, 0], 'umap_2': coords[:, 1], 'label': y}).to_csv(umap_csv_path, index=false)\n    print(f'saved: {umap_png_path}')\nelse:\n    print('[warning] the rerun/training flags are false and the saved artifacts for this section are missing. skipping this section.')"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Save Outputs

In [ ]:
try:
    if memory_bank is not None:
        torch.save({'extractor_state': extractor.state_dict(), 'memory_bank': memory_bank.cpu(), 'threshold': float(best_thresh), 'threshold_quantile': float(THRESHOLD_QUANTILE), 'config': dict(backbone=BACKBONE_NAME, vit_feature_block=VIT_FEATURE_BLOCK, patch_embed_dim=PATCH_EMBED_DIM, image_size=IMAGE_SIZE, train_normal_n=TRAIN_NORMAL_N, tune_normal_n=TUNE_NORMAL_N, test_normal_n=TEST_NORMAL_N, test_defect_n=TEST_DEFECT_N, memory_bank_max=MEMORY_BANK_MAX, patchcore_nn_k=PATCHCORE_NN_K, topk_patch_ratio=TOPK_PATCH_RATIO, threshold_quantile=THRESHOLD_QUANTILE)}, MODEL_EXPORT_PATH)
        print(f'Model saved -> {MODEL_EXPORT_PATH}')
    else:
        print('Scores reused from disk - model artifact unchanged.')
    metrics = dict(roc_auc=float(roc_auc), auprc=float(auprc), f1_defect=float(f1_val), precision_defect=float(precision_val), recall_defect=float(recall_val), threshold=float(best_thresh), threshold_quantile=float(THRESHOLD_QUANTILE), confusion_matrix=cm.tolist(), n_test_normal=int(len(test_normal_scores)), n_test_defect=int(len(test_defect_scores)), per_class=perclass_results)
    with open(METRICS_EXPORT_PATH, 'w') as fh:
        json.dump(metrics, fh, indent=2)
    summary = dict(name=f'PatchCore DINOv2 ViT-B/14 block-{VIT_FEATURE_BLOCK}', checkpoint_path=MODEL_EXPORT_PATH, metrics=metrics, config=dict(backbone=BACKBONE_NAME, vit_feature_block=VIT_FEATURE_BLOCK, patch_embed_dim=PATCH_EMBED_DIM, memory_bank_max=MEMORY_BANK_MAX, patchcore_nn_k=PATCHCORE_NN_K, topk_patch_ratio=TOPK_PATCH_RATIO, threshold_quantile=THRESHOLD_QUANTILE))
    with open(SUMMARY_EXPORT_PATH, 'w') as fh:
        json.dump(summary, fh, indent=2)
    print(f'Metrics -> {METRICS_EXPORT_PATH}')
    print(f'Summary -> {SUMMARY_EXPORT_PATH}')
    print(f'\nROC-AUC : {roc_auc:.4f}')
    print(f'AUPRC   : {auprc:.4f}')
    print(f'F1      : {f1_val:.4f}')
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "if memory_bank is not none:\n    torch.save({'extractor_state': extractor.state_dict(), 'memory_bank': memory_bank.cpu(), 'threshold': float(best_thresh), 'threshold_quantile': float(threshold_quantile), 'config': dict(backbone=backbone_name, vit_feature_block=vit_feature_block, patch_embed_dim=patch_embed_dim, image_size=image_size, train_normal_n=train_normal_n, tune_normal_n=tune_normal_n, test_normal_n=test_normal_n, test_defect_n=test_defect_n, memory_bank_max=memory_bank_max, patchcore_nn_k=patchcore_nn_k, topk_patch_ratio=topk_patch_ratio, threshold_quantile=threshold_quantile)}, model_export_path)\n    print(f'model saved -> {model_export_path}')\nelse:\n    print('scores reused from disk - model artifact unchanged.')\nmetrics = dict(roc_auc=float(roc_auc), auprc=float(auprc), f1_defect=float(f1_val), precision_defect=float(precision_val), recall_defect=float(recall_val), threshold=float(best_thresh), threshold_quantile=float(threshold_quantile), confusion_matrix=cm.tolist(), n_test_normal=int(len(test_normal_scores)), n_test_defect=int(len(test_defect_scores)), per_class=perclass_results)\nwith open(metrics_export_path, 'w') as fh:\n    json.dump(metrics, fh, indent=2)\nsummary = dict(name=f'patchcore dinov2 vit-b/14 block-{vit_feature_block}', checkpoint_path=model_export_path, metrics=metrics, config=dict(backbone=backbone_name, vit_feature_block=vit_feature_block, patch_embed_dim=patch_embed_dim, memory_bank_max=memory_bank_max, patchcore_nn_k=patchcore_nn_k, topk_patch_ratio=topk_patch_ratio, threshold_quantile=threshold_quantile))\nwith open(summary_export_path, 'w') as fh:\n    json.dump(summary, fh, indent=2)\nprint(f'metrics -> {metrics_export_path}')\nprint(f'summary -> {summary_export_path}')\nprint(f'\\nroc-auc : {roc_auc:.4f}')\nprint(f'auprc   : {auprc:.4f}')\nprint(f'f1      : {f1_val:.4f}')"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Cleanup

In [ ]:
for name in ['extractor', 'memory_bank', 'tune_normal_scores', 'test_normal_scores', 'test_defect_scores', 'all_scores', 'all_labels', 'predictions', 'train_loader', 'tune_normal_loader', 'test_normal_loader', 'test_defect_loader']:
    if name in globals():
        del globals()[name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
print('Cleared.')